In [1]:
import json
import os

In [2]:
with open('../JSON Files/new_forward_index.json') as f:
    new_forward_index = json.load(f)
print(f'New forward index loaded with {len(new_forward_index)} documents')

New forward index loaded with 1 documents


In [3]:
# Initializing inverted index
inverted_index = {}

In [4]:
# Iterate through each document in the forward index
for doc_id, fields in new_forward_index.items():
    for field, word_ids in fields.items():  # 'field' can be "title" or "text"
        for word_id in word_ids:  # Iterate over word IDs in each field
            if word_id not in inverted_index:
                inverted_index[word_id] = {"df": 0, "postings": {}}
            
            if doc_id not in inverted_index[word_id]["postings"]:
                inverted_index[word_id]["postings"][doc_id] = {"tf": 0, "positions": {"title": 0, "text": 0}}
                inverted_index[word_id]["df"] += 1  
            
            # Increment term frequency and track positions in the respective field
            inverted_index[word_id]["postings"][doc_id]["tf"] += 1
            inverted_index[word_id]["postings"][doc_id]["positions"][field] += 1  # Increment title/text position count
        

In [5]:
print(inverted_index)

{108193: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 1, 'text': 0}}}}, 485092: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 0, 'text': 1}}}}, 485093: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 0, 'text': 1}}}}, 485094: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 0, 'text': 1}}}}, 4441: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 0, 'text': 1}}}}, 444694: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 0, 'text': 1}}}}, 485095: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 0, 'text': 1}}}}, 63348: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 0, 'text': 1}}}}, 485096: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 0, 'text': 1}}}}, 485097: {'df': 1, 'postings': {'doc192362': {'tf': 1, 'positions': {'title': 0, 'text': 1}}}}}


In [7]:
def get_barrel(word_id):
    # For barrel_10_1 to barrel_10_1000 (First 10000 words)
    if word_id < 10000:
        barrel_index = (word_id // 10) + 1
        return f'../JSON Files/Barrels/barrel_10/barrel_10_{barrel_index}.json'
    
    # For barrel_250_1 to barrel_250_80 (Next 20000 words)
    elif word_id >= 10000 and word_id < 30000:
        barrel_index = ((word_id-10000) // 250) + 1
        return f'../JSON Files/Barrels/barrel_250/barrel_250_{barrel_index}.json'
    
    # For barrel_10000_1 to barrel_10000_46 (Remaining words)
    else:
        barrel_index = (word_id // 10000) - 2
        return f'../JSON Files/Barrels/barrel_10000/barrel_10000_{barrel_index}.json'

In [8]:
# Function to add the new inverted index words to the respective barrel
def update_barrel(word_id, inverted_data): 
    barrel_file = get_barrel(word_id)

    if os.path.exists(barrel_file):
        with open(barrel_file) as f:
            barrel_data = json.load(f)
        print(f'{barrel_file} loaded with {len(barrel_data)} words')
    else:
        print(f"{barrel_file} does not exist, creating a new barrel.")
        barrel_data = {}

    
    # Checking if the word already exists in the barrel
    word_id = str(word_id)
    if word_id in barrel_data.keys():
        print(f"Word ID {word_id} already exists in the barrel.")
        barrel_data[word_id]['df'] += inverted_data['df']

        for doc_id, posting in inverted_data['postings'].items():
            print(f"Updating doc ID: {doc_id}")
            barrel_data[word_id]['postings'][doc_id] = posting
    else:
        print(f"Word ID {word_id} does not exist, adding to the barrel.")
        barrel_data[word_id] = inverted_data

    # Save the updated barrel data back to the file
    with open(barrel_file, 'w') as f:
        json.dump(barrel_data, f)
    print(f'{barrel_file} updated with {len(barrel_data)} words')

In [9]:
for word_id, inverted_data in inverted_index.items():
    update_barrel(word_id, inverted_data)

print("Barrels updated successfully.")

../JSON Files/Barrels/barrel_10000/barrel_10000_8.json loaded with 10000 words
Word ID 108193 does not exist, adding to the barrel.
../JSON Files/Barrels/barrel_10000/barrel_10000_8.json updated with 10001 words
../JSON Files/Barrels/barrel_10000/barrel_10000_46.json loaded with 10000 words
Word ID 485092 does not exist, adding to the barrel.
../JSON Files/Barrels/barrel_10000/barrel_10000_46.json updated with 10001 words
../JSON Files/Barrels/barrel_10000/barrel_10000_46.json loaded with 10001 words
Word ID 485093 does not exist, adding to the barrel.
../JSON Files/Barrels/barrel_10000/barrel_10000_46.json updated with 10002 words
../JSON Files/Barrels/barrel_10000/barrel_10000_46.json loaded with 10002 words
Word ID 485094 does not exist, adding to the barrel.
../JSON Files/Barrels/barrel_10000/barrel_10000_46.json updated with 10003 words
../JSON Files/Barrels/barrel_10/barrel_10_445.json loaded with 10 words
Word ID 4441 already exists in the barrel.
Updating doc ID: doc192362
../J

<hr>